# Quant System Proof of Concept (POC)
This notebook demonstrates the complete workflow of the quantitative trading system.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to sys.path
project_root = str(Path(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

from data_infra.fetchers import TushareFetcher
from data_infra.storage import StorageManager
import pandas as pd
import qlib
from qlib.constant import REG_CN
from qlib.utils import init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord

## 1. Data Fetching and Storage

In [ ]:
# Initialize fetcher and storage
fetcher = TushareFetcher(config_path="config.yaml")
storage = StorageManager()

# Note: Replace these dates with a wider range to get meaningful model results
trade_dates = ['20230103', '20230104', '20230105', '20230106', '20230109'] 
for date in trade_dates:
    print(f"Fetching data for {date}...")
    try:
        df = fetcher.fetch_daily_data(trade_date=date)
        if df is not None and not df.empty:
            storage.insert_daily_data(df)
    except Exception as e:
        print(f"Error fetching data for {date}: {e}")

In [ ]:
# Export cached raw data to Qlib format
qlib_data_dir = os.path.join(project_root, "data", "qlib_data")
os.makedirs(qlib_data_dir, exist_ok=True)

print("Exporting to Qlib format...")
storage.export_to_qlib(dest_dir=qlib_data_dir)
print("Data export complete.")

## 2. Initialize Qlib and Define Alpha/Model Pipeline

In [ ]:
# Initialize Qlib
qlib.init(provider_uri=qlib_data_dir, region=REG_CN)

# Define dataset, model, and workflow configuration
market = "all"
benchmark = "SH000300"

dataset_config = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "DataHandlerLP",
            "module_path": "qlib.data.dataset.handler",
            "kwargs": {
                "start_time": "2023-01-01",
                "end_time": "2023-01-15",
                "instruments": market,
                "data_loader": {
                    "class": "QlibDataLoader",
                    "module_path": "qlib.data.dataset.loader",
                    "kwargs": {
                        "config": {
                            "feature": (
                                ["$close/Ref($close, 1)-1", "$high/$low"],
                                ["return", "high_low_ratio"]
                            ),
                            "label": (
                                ["Ref($close, -1) / $close - 1"],
                                ["label"]
                            )
                        }
                    }
                }
            }
        },
        "segments": {
            "train": ("2023-01-01", "2023-01-05"),
            "valid": ("2023-01-06", "2023-01-09"),
            "test": ("2023-01-10", "2023-01-15"),
        }
    }
}

model_config = {
    "class": "LGBModel",
    "module_path": "qlib.contrib.model.gbdt",
    "kwargs": {
        "loss": "mse",
        "colsample_bytree": 0.8879,
        "learning_rate": 0.0421,
        "subsample": 0.8789,
        "lambda_l1": 205.6999,
        "lambda_l2": 580.9768,
        "max_depth": 8,
        "num_leaves": 210,
        "num_threads": 20,
    }
}

port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True
        }
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "topk": 5,
            "n_drop": 1
        }
    },
    "backtest": {
        "start_time": "2023-01-10",
        "end_time": "2023-01-15",
        "account": 100000000,
        "benchmark": benchmark,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5
        }
    }
}

## 3. Train Model and Run Backtest

In [ ]:
# Load dataset
dataset = init_instance_by_config(dataset_config)

# Start MLflow run (or Qlib default tracking)
with R.start(experiment_name="poc_experiment"):
    # Initialize and train model
    model = init_instance_by_config(model_config)
    print("Training model...")
    model.fit(dataset)
    R.save_objects(trained_model=model)
    print("Model training complete.")

    print("Generating prediction signals...")
    # Generate and save predictions (Signal Record)
    sr = SignalRecord(model, dataset)
    sr.generate()

    print("Running backtest...")
    # Run portfolio analysis (Backtest)
    par = PortAnaRecord(R.get_recorder(), port_analysis_config, "day")
    par.generate()

print("Backtest completed. Check MLflow or local qlib tracking folder for results.")